In [29]:
### Replace ReLU with Sigmoid in the hidden layers. How does training speed change ###

In [30]:
# import libraries

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import time

In [31]:
# Load MNIST
mnist = fetch_openml('mnist_784', version=1)
# Convert to NumPy
X = mnist.data.to_numpy() / 255.0
y = mnist.target.astype(int).to_numpy()

In [32]:
# One-hot encode labels

encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y.reshape(-1,1))

In [33]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [34]:
# Activation functions
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return x > 0

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

In [35]:
# Softmax
def softmax(x):
    exp = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

In [36]:
# Initialize weights
def init_weights():
    W1 = np.random.randn(784,128) * 0.01
    W2 = np.random.randn(128,64) * 0.01
    W3 = np.random.randn(64,10) * 0.01
    return W1,W2,W3

In [38]:
def train_model(activation, activation_derivative, name):

    W1,W2,W3 = init_weights()
    lr = 0.1
    epochs = 5

    start_total = time.time()

    for epoch in range(epochs):

        start_epoch = time.time()

        # Forward pass
        Z1 = X_train @ W1
        A1 = activation(Z1)

        Z2 = A1 @ W2
        A2 = activation(Z2)

        Z3 = A2 @ W3
        A3 = softmax(Z3)

        # Loss
        loss = -np.mean(y_train * np.log(A3 + 1e-8))

        # Backprop
        dZ3 = A3 - y_train
        dW3 = A2.T @ dZ3 / X_train.shape[0]

        dZ2 = (dZ3 @ W3.T) * activation_derivative(Z2)
        dW2 = A1.T @ dZ2 / X_train.shape[0]

        dZ1 = (dZ2 @ W2.T) * activation_derivative(Z1)
        dW1 = X_train.T @ dZ1 / X_train.shape[0]

        # Update
        W1 -= lr * dW1
        W2 -= lr * dW2
        W3 -= lr * dW3

        epoch_time = time.time() - start_epoch

        print(f"{name} Epoch {epoch+1} Loss {loss:.4f} Time {epoch_time:.3f}s")

    total_time = time.time() - start_total

    print(f"\n{name} Total Training Time: {total_time:.3f}s\n")

In [ ]:
print("Training ReLU Model")
train_model(relu, relu_derivative, "ReLU")

print("\nTraining Sigmoid Model")
train_model(sigmoid, sigmoid_derivative, "Sigmoid")